# 00 Dataset Check

## 1. Notebook purpose and workflow

This notebook verifies the frozen model-ready dataset package for the thesis workflow:

**Genetic Algorithm-Evolved Feature-Group Importance Weighting Fused with Frozen DistilBERT Embeddings for Adversarially Robust English Smishing Detection**

Workflow:

1. Manually upload `06_model_ready.zip` to the Colab sidebar.
2. Unzip it into `/content/`.
3. Run dataset checks only. No model training happens here.
4. Save reports locally under `/content/thesis_outputs/reports/`.
5. Zip `/content/thesis_outputs/` into `/content/thesis_outputs.zip`.
6. Manually download `thesis_outputs.zip` and upload it to Google Drive or GitHub yourself.

**Important:** This notebook does not mount Google Drive and does not save anything to Google Drive.

## 2. Imports

In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

TEXT_COL = "message"
LABEL_COL = "final_label"

CONTENT_DIR = Path("/content")
ZIP_PATH = CONTENT_DIR / "06_model_ready.zip"
OUTPUT_DIR = CONTENT_DIR / "thesis_outputs"
REPORTS_DIR = OUTPUT_DIR / "reports"
OUTPUT_ZIP = CONTENT_DIR / "thesis_outputs.zip"

required_folders = [
    "clean",
    "augmented_training",
    "adversarial_validation",
    "adversarial_test",
    "manifests",
    "reports",
]

csv_files = {
    "final_clean": "clean/final_clean_dataset.csv",
    "train_clean": "clean/train_clean.csv",
    "val_clean": "clean/val_clean.csv",
    "test_clean": "clean/test_clean.csv",
    "train_augmented_b": "augmented_training/train_augmented_for_ablation_b.csv",
    "val_adv_10": "adversarial_validation/val_adv_10.csv",
    "val_adv_20": "adversarial_validation/val_adv_20.csv",
    "test_adv_10": "adversarial_test/test_adv_10.csv",
    "test_adv_20": "adversarial_test/test_adv_20.csv",
    "test_adv_30": "adversarial_test/test_adv_30.csv",
}

expected_rows = {
    "final_clean": 10544,
    "train_clean": 7380,
    "val_clean": 1582,
    "test_clean": 1582,
    "train_augmented_b": 9580,
    "val_adv_10": 1582,
    "val_adv_20": 1582,
    "test_adv_10": 1582,
    "test_adv_20": 1582,
    "test_adv_30": 1582,
}

expected_label_counts = {
    "final_clean": {"ham": 5272, "smishing": 5272},
    "train_clean": {"ham": 3690, "smishing": 3690},
    "val_clean": {"ham": 791, "smishing": 791},
    "test_clean": {"ham": 791, "smishing": 791},
    "train_augmented_b": {"ham": 3690, "smishing": 5890},
    "val_adv_10": {"ham": 791, "smishing": 791},
    "val_adv_20": {"ham": 791, "smishing": 791},
    "test_adv_10": {"ham": 791, "smishing": 791},
    "test_adv_20": {"ham": 791, "smishing": 791},
    "test_adv_30": {"ham": 791, "smishing": 791},
}

print("Imports complete.")

## 3. Locate uploaded ZIP file at `/content/06_model_ready.zip`

In [ ]:
print(f"Looking for uploaded ZIP at: {ZIP_PATH}")
assert ZIP_PATH.exists(), (
    "Missing /content/06_model_ready.zip. Upload 06_model_ready.zip to the Colab sidebar, "
    "then rerun this cell."
)
assert ZIP_PATH.is_file(), f"Expected a file, but found something else at {ZIP_PATH}"
print(f"Found ZIP: {ZIP_PATH} ({ZIP_PATH.stat().st_size:,} bytes)")

## 4. Unzip the ZIP file

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_member = zf.testzip()
    assert bad_member is None, f"ZIP integrity check failed at member: {bad_member}"
    zf.extractall(CONTENT_DIR)

print(f"Extracted {ZIP_PATH.name} into {CONTENT_DIR}")

## 5. Automatically locate the extracted `06_model_ready` folder

In [ ]:
def locate_model_ready_folder(content_dir: Path) -> Path:
    candidates = [
        content_dir / "06_model_ready",
        content_dir / "model_ready",
        content_dir / "thesis-modeling" / "data" / "06_model_ready",
        content_dir / "data" / "06_model_ready",
    ]

    for candidate in candidates:
        if candidate.exists() and candidate.is_dir():
            return candidate

    matches = [p for p in content_dir.rglob("06_model_ready") if p.is_dir()]
    if matches:
        return sorted(matches, key=lambda p: len(str(p)))[0]

    raise FileNotFoundError(
        "Could not locate the extracted 06_model_ready folder. Checked the expected locations "
        "and searched under /content/."
    )


DATA_DIR = locate_model_ready_folder(CONTENT_DIR)
print(f"Using dataset folder: {DATA_DIR}")

## 6. Create local output folder

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output folder: {OUTPUT_DIR}")
print(f"Reports folder: {REPORTS_DIR}")

## 7. Check required folders

In [ ]:
folder_check = pd.DataFrame(
    [
        {
            "folder": folder,
            "path": str(DATA_DIR / folder),
            "exists": (DATA_DIR / folder).exists(),
            "is_dir": (DATA_DIR / folder).is_dir(),
        }
        for folder in required_folders
    ]
)
folder_check["passed"] = folder_check["exists"] & folder_check["is_dir"]
folder_check.to_csv(REPORTS_DIR / "00_folder_check.csv", index=False)
folder_check

## 8. Check required CSV files

In [ ]:
file_check = pd.DataFrame(
    [
        {
            "dataset": name,
            "relative_path": rel_path,
            "path": str(DATA_DIR / rel_path),
            "exists": (DATA_DIR / rel_path).exists(),
            "is_file": (DATA_DIR / rel_path).is_file(),
        }
        for name, rel_path in csv_files.items()
    ]
)
file_check["passed"] = file_check["exists"] & file_check["is_file"]
file_check.to_csv(REPORTS_DIR / "00_file_check.csv", index=False)
file_check

## 9. Stop with clear error if anything is missing

In [ ]:
missing_folders = folder_check.loc[~folder_check["passed"], "folder"].tolist()
missing_files = file_check.loc[~file_check["passed"], "relative_path"].tolist()

assert not missing_folders, f"Missing required folders inside {DATA_DIR}: {missing_folders}"
assert not missing_files, f"Missing required CSV files inside {DATA_DIR}: {missing_files}"

print("All required folders and CSV files are present.")

## 10. Load all CSV files with pandas

In [ ]:
dfs = {}

for name, rel_path in csv_files.items():
    path = DATA_DIR / rel_path
    dfs[name] = pd.read_csv(path)
    print(f"Loaded {name}: {dfs[name].shape[0]:,} rows x {dfs[name].shape[1]:,} columns")

## 11. Preview shapes, columns, and first few rows

In [ ]:
for name, df in dfs.items():
    print("=" * 80)
    print(name)
    print(f"Shape: {df.shape}")
    print("Columns:", list(df.columns))
    display(df.head())

## 12. Set and validate text/label columns

Defaults are `TEXT_COL = "message"` and `LABEL_COL = "final_label"`. If this cell fails, it prints every dataset's columns so you can update the two constants in the imports cell.

In [ ]:
def print_all_columns(datasets: dict[str, pd.DataFrame]) -> None:
    print("Available columns by dataset:")
    for dataset_name, frame in datasets.items():
        print(f"\n{dataset_name}:")
        for column in frame.columns:
            print(f"  - {column}")


missing_column_messages = []
for name, df in dfs.items():
    if TEXT_COL not in df.columns:
        missing_column_messages.append(f"{name} is missing text column {TEXT_COL!r}")
    if LABEL_COL not in df.columns:
        missing_column_messages.append(f"{name} is missing label column {LABEL_COL!r}")

if missing_column_messages:
    print_all_columns(dfs)

assert not missing_column_messages, "Column validation failed: " + "; ".join(missing_column_messages)
print(f"Text column validated: {TEXT_COL!r}")
print(f"Label column validated: {LABEL_COL!r}")

## 13. Check expected row counts

In [ ]:
row_count_check = pd.DataFrame(
    [
        {
            "dataset": name,
            "expected_rows": expected_rows[name],
            "actual_rows": len(dfs[name]),
            "passed": len(dfs[name]) == expected_rows[name],
        }
        for name in expected_rows
    ]
)
row_count_check.to_csv(REPORTS_DIR / "00_row_count_check.csv", index=False)
row_count_check

## 14. Assert row counts

In [ ]:
failed_row_counts = row_count_check.loc[~row_count_check["passed"]].to_dict(orient="records")
assert not failed_row_counts, f"Row count check failed: {failed_row_counts}"
print("All row counts match expected values.")

## 15. Check label distribution

In [ ]:
label_summary_records = []

for name, df in dfs.items():
    counts = df[LABEL_COL].value_counts(dropna=False).to_dict()
    for label, count in counts.items():
        label_summary_records.append(
            {
                "dataset": name,
                "label": "<NA>" if pd.isna(label) else str(label),
                "count": int(count),
            }
        )

label_summary = pd.DataFrame(label_summary_records).sort_values(["dataset", "label"])
label_summary.to_csv(REPORTS_DIR / "00_label_summary.csv", index=False)
label_summary

## 16. Assert expected label counts

In [ ]:
label_count_records = []

for dataset_name, expected_counts in expected_label_counts.items():
    actual_counts = dfs[dataset_name][LABEL_COL].value_counts(dropna=False).to_dict()
    for label_name, expected_count in expected_counts.items():
        actual_count = int(actual_counts.get(label_name, 0))
        label_count_records.append(
            {
                "dataset": dataset_name,
                "label": label_name,
                "expected_count": expected_count,
                "actual_count": actual_count,
                "passed": actual_count == expected_count,
            }
        )

label_count_check = pd.DataFrame(label_count_records)
label_count_check.to_csv(REPORTS_DIR / "00_label_count_check.csv", index=False)
display(label_count_check)

failed_label_counts = label_count_check.loc[~label_count_check["passed"]].to_dict(orient="records")
assert not failed_label_counts, f"Label count check failed: {failed_label_counts}"
print("All label counts match expected values.")

## 17. Check missing text and missing labels

In [ ]:
quality_records = []

for name, df in dfs.items():
    text_as_string = df[TEXT_COL].astype("string")
    missing_text = int(df[TEXT_COL].isna().sum() + text_as_string.str.strip().eq("").fillna(False).sum())
    missing_labels = int(df[LABEL_COL].isna().sum())
    duplicate_text_rows = int(df.duplicated(subset=[TEXT_COL], keep=False).sum())
    unique_texts = int(df[TEXT_COL].nunique(dropna=True))

    quality_records.append(
        {
            "dataset": name,
            "rows": len(df),
            "missing_text_or_blank_text": missing_text,
            "missing_labels": missing_labels,
            "duplicate_text_rows": duplicate_text_rows,
            "unique_texts": unique_texts,
            "missing_text_passed": missing_text == 0,
            "missing_label_passed": missing_labels == 0,
        }
    )

quality_check = pd.DataFrame(quality_records)
quality_check.to_csv(REPORTS_DIR / "00_quality_check.csv", index=False)
quality_check

## 18. Check duplicate text rows

Duplicate text rows are reported for review. This notebook does not automatically fail on duplicates because augmented/adversarial files may intentionally contain related or repeated text depending on the generation process.

In [ ]:
duplicate_summary = quality_check[["dataset", "duplicate_text_rows", "unique_texts", "rows"]]
duplicate_summary

## 19. Assert no missing text or labels

In [ ]:
missing_quality_failures = quality_check.loc[
    ~(quality_check["missing_text_passed"] & quality_check["missing_label_passed"])
].to_dict(orient="records")

assert not missing_quality_failures, f"Missing text/label check failed: {missing_quality_failures}"
print("No missing or blank text values, and no missing labels, were found.")

## 20. Check exact text leakage across clean train/val/test splits

In [ ]:
def text_set(dataset_name: str) -> set:
    return set(dfs[dataset_name][TEXT_COL].dropna().astype(str))


split_pairs = [
    ("train_clean", "val_clean"),
    ("train_clean", "test_clean"),
    ("val_clean", "test_clean"),
]

leakage_check = {}

for left_name, right_name in split_pairs:
    overlap = sorted(text_set(left_name) & text_set(right_name))
    pair_name = f"{left_name}_vs_{right_name}"
    leakage_check[pair_name] = {
        "left_dataset": left_name,
        "right_dataset": right_name,
        "overlap_count": len(overlap),
        "sample_overlaps": overlap[:20],
        "passed": len(overlap) == 0,
    }

with open(REPORTS_DIR / "00_leakage_check.json", "w", encoding="utf-8") as f:
    json.dump(leakage_check, f, indent=2, ensure_ascii=False)

leakage_check

## 21. Assert no clean split leakage

In [ ]:
leakage_failures = {
    pair_name: result
    for pair_name, result in leakage_check.items()
    if not result["passed"]
}

assert not leakage_failures, f"Clean split text leakage found: {leakage_failures}"
print("No exact text leakage found across clean train/val/test splits.")

## 22. Check adversarial validation/test files have same row counts as clean val/test

In [ ]:
adversarial_size_pairs = [
    ("val_adv_10", "val_clean"),
    ("val_adv_20", "val_clean"),
    ("test_adv_10", "test_clean"),
    ("test_adv_20", "test_clean"),
    ("test_adv_30", "test_clean"),
]

adversarial_size_check = pd.DataFrame(
    [
        {
            "adversarial_dataset": adv_name,
            "reference_clean_dataset": clean_name,
            "adversarial_rows": len(dfs[adv_name]),
            "reference_clean_rows": len(dfs[clean_name]),
            "passed": len(dfs[adv_name]) == len(dfs[clean_name]),
        }
        for adv_name, clean_name in adversarial_size_pairs
    ]
)
adversarial_size_check.to_csv(REPORTS_DIR / "00_adversarial_size_check.csv", index=False)
adversarial_size_check

## 23. Assert adversarial size checks

In [ ]:
failed_adversarial_sizes = adversarial_size_check.loc[~adversarial_size_check["passed"]].to_dict(orient="records")
assert not failed_adversarial_sizes, f"Adversarial size check failed: {failed_adversarial_sizes}"
print("All adversarial validation/test files match the row counts of their clean references.")

## 24. Save CSV/JSON reports locally

In [ ]:
report_paths = sorted(REPORTS_DIR.glob("00_*"))

print("Reports saved locally:")
for report_path in report_paths:
    print(f"- {report_path}")

## 25. Create combined summary JSON

In [ ]:
checks = {
    "folder_check_passed": bool(folder_check["passed"].all()),
    "file_check_passed": bool(file_check["passed"].all()),
    "row_count_check_passed": bool(row_count_check["passed"].all()),
    "label_count_check_passed": bool(label_count_check["passed"].all()),
    "missing_text_check_passed": bool(quality_check["missing_text_passed"].all()),
    "missing_label_check_passed": bool(quality_check["missing_label_passed"].all()),
    "clean_split_leakage_check_passed": bool(all(result["passed"] for result in leakage_check.values())),
    "adversarial_size_check_passed": bool(adversarial_size_check["passed"].all()),
}

summary = {
    "notebook": "00_dataset_check.ipynb",
    "dataset_folder": str(DATA_DIR),
    "zip_path": str(ZIP_PATH),
    "output_dir": str(OUTPUT_DIR),
    "reports_dir": str(REPORTS_DIR),
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "checks": checks,
    "all_checks_passed": bool(all(checks.values())),
    "row_counts": row_count_check.to_dict(orient="records"),
    "quality": quality_check.to_dict(orient="records"),
    "leakage": leakage_check,
}

summary_path = REPORTS_DIR / "00_dataset_check_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Combined summary saved to: {summary_path}")
summary

## 26. Print final pass/fail message

In [ ]:
if summary["all_checks_passed"]:
    print("✅ DATASET CHECK PASSED")
    print("You can proceed to 01_tfidf_logreg_baselines.ipynb.")
else:
    print("❌ DATASET CHECK FAILED")
    failed_checks = [name for name, passed in checks.items() if not passed]
    print("Failed checks:", failed_checks)

assert summary["all_checks_passed"], "Dataset check failed. Review the reports in /content/thesis_outputs/reports/."

## 27. Zip `/content/thesis_outputs/` into `/content/thesis_outputs.zip`

In [ ]:
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

shutil.make_archive(str(OUTPUT_ZIP.with_suffix("")), "zip", root_dir=OUTPUT_DIR)

assert OUTPUT_ZIP.exists(), f"Failed to create {OUTPUT_ZIP}"
print(f"Created local output ZIP: {OUTPUT_ZIP}")
print("Download thesis_outputs.zip from the Colab sidebar, then upload it wherever you prefer.")